# Day 24 — PhysioNet 데이터 접근과 구조: 데이터 사전 만들기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/week-4/day-24.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/week-4/day-24.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1: JSON 메타데이터를 표로 펼치기

아래 JSON은 PADS 페이지에 실린 `observation_001.json` 발췌를 그대로 옮긴 것입니다(일부 항목 생략). 실제 파일을 받기 전에도 구조를 연습할 수 있습니다.

In [ ]:
import json
import pandas as pd

# PhysioNet PADS 페이지에 실린 observation JSON 발췌를 그대로 옮긴 것(일부만)
text = """
{
  "resource_type": "observation",
  "subject_id": "001",
  "study_id": "PADS",
  "device_id": "Apple Watch Series 4",
  "sampling_rate": 100,
  "session": [
    {
      "record_name": "Relaxed",
      "rows": 2048,
      "records": [
        {"device_location": "LeftWrist",
         "channels": ["Accelerometer_X", "Accelerometer_Y", "Accelerometer_Z",
                      "Gyroscope_X", "Gyroscope_Y", "Gyroscope_Z"],
         "units": ["g", "g", "g", "rad/s", "rad/s", "rad/s"],
         "file_name": "bins/001_Relaxed_LeftWrist.bin"},
        {"device_location": "RightWrist",
         "channels": ["Accelerometer_X", "Accelerometer_Y", "Accelerometer_Z",
                      "Gyroscope_X", "Gyroscope_Y", "Gyroscope_Z"],
         "units": ["g", "g", "g", "rad/s", "rad/s", "rad/s"],
         "file_name": "bins/001_Relaxed_RightWrist.bin"}
      ]
    }
  ]
}
"""
obs = json.loads(text)        # 실제 파일이라면: obs = json.load(open(경로))

print("최상위 key:", list(obs.keys()))
print("sampling rate:", obs["sampling_rate"], "Hz")

rows = []
for session in obs["session"]:              # 과제 하나
    for rec in session["records"]:          # 손목 하나
        rows.append({
            "subject_id": obs["subject_id"],
            "task": session["record_name"],
            "wrist": rec["device_location"],
            "n_rows": session["rows"],
            "seconds": session["rows"] / obs["sampling_rate"],
            "n_channels": len(rec["channels"]),
        })
print(pd.DataFrame(rows).to_string(index=False))

### 예시 2: 머리글 없는 신호 파일의 처음 몇 줄 읽기

아래 세 줄은 실제 `001_Relaxed_LeftWrist.txt`의 처음 세 줄입니다. 머리글이 없으므로 열 이름을 **내가** 붙여야 합니다.

In [ ]:
import io
import pandas as pd

# PADS 001_Relaxed_LeftWrist.txt의 처음 3줄(실제 파일에서 확인한 값)
sample = """0.0000000000,-0.0039583882,0.0023599046,0.0017461196,-0.0154924262,0.0056162034,0.0010342058
0.0099029541,-0.0040048202,0.0022586959,0.0008355394,-0.0122763924,0.0056047519,-0.0032175307
0.0199010372,0.0008331696,0.0022254365,0.0009437218,-0.0090513043,0.0024323943,-0.0021368200
"""
cols = ["time_s", "acc_x", "acc_y", "acc_z", "gyr_x", "gyr_y", "gyr_z"]   # 내가 붙인 이름
df = pd.read_csv(io.StringIO(sample), header=None, names=cols)
# 실제 파일이라면: pd.read_csv("<경로>/001_Relaxed_LeftWrist.txt", header=None, names=cols)

print(df.shape)
print(df.round(4).to_string(index=False))
print("시간 간격(초):", df["time_s"].diff().round(4).tolist())

## 직접 해보기

1. 두 데이터셋 페이지를 브라우저로 엽니다. 각 페이지에서 **Access Policy, License, DOI, 버전, 공개일, 전체 크기**를 찾아 `reports/physionet-pd-data-dictionary.md` 맨 위 "출처" 표에 적습니다. 인용 문구(페이지의 "please cite")도 복사해 둡니다. 오늘 날짜를 "확인일"로 적습니다.
2. 둘 중 **하나**를 고르고, 고른 이유를 두 문장으로 씁니다.
3. 데이터를 받습니다. 전체를 받기 전에 **작은 파일 몇 개만** 받아 구조를 봅니다.
   - PADS: 브라우저의 Files 영역에서 `preprocessed/file_list.csv`, `patients/patient_001.json`, `movement/observation_001.json`, `movement/timeseries/` 안의 파일 2개를 받아 `data/physionet_pd/pads/`에 원래 폴더 구조 그대로 둡니다.
   - Gait: `format.txt`, `demographics.txt`, `GaCo01_01.txt`, `GaPt03_01.txt`를 받아 `data/physionet_pd/gaitpdb/`에 둡니다.
   - 전체가 필요해지면 페이지에 적힌 명령을 씁니다(PADS는 1.4 GB이니 저장 공간을 확인하세요):

In [ ]:
# 내 컴퓨터용 명령입니다. Colab에서는 의미가 없어 실행하지 않도록 주석 처리했습니다.
# wget -r -N -c -np https://physionet.org/files/parkinsons-disease-smartwatch/1.0.0/
# wget -r -N -c -np https://physionet.org/files/gaitpdb/1.0.0/

4. 받은 파일을 **텍스트 편집기로 먼저 열어** 처음 5줄을 눈으로 봅니다. 구분자가 쉼표인지 탭인지, 머리글이 있는지, 열이 몇 개인지 적습니다. 그다음 Python으로 확인합니다.

In [ ]:
import pandas as pd
path = "<받은 파일 경로>"                       # 자리표시자: 내 경로로 바꾸기
df = pd.read_csv(path, header=None, sep=None, engine="python")   # 구분자 자동 추정
print(df.shape)
print(df.head())